<a href="https://colab.research.google.com/github/adi-asu-git/FSE570-Alaska/blob/main/brfss_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!ls /content/drive/MyDrive/Capstone

 brfss_preprocess.ipynb
 capstone_phase1_exploration.ipynb
 final_merge.csv
'LLCP2021.XPT '
'PLACES__ZCTA_Data_(GIS_Friendly_Format),_2024_release_20260306.csv'


In [11]:
!mv "/content/drive/MyDrive/Capstone/LLCP2021.XPT " "/content/drive/MyDrive/Capstone/LLCP2021.XPT"

In [12]:
import pandas as pd

brfss = pd.read_sas('/content/drive/MyDrive/Capstone/LLCP2021.XPT', format='xport')

brfss.shape

(438693, 303)

In [13]:
brfss.columns.tolist()

['_STATE',
 'FMONTH',
 'IDATE',
 'IMONTH',
 'IDAY',
 'IYEAR',
 'DISPCODE',
 'SEQNO',
 '_PSU',
 'CTELENM1',
 'PVTRESD1',
 'COLGHOUS',
 'STATERE1',
 'CELPHON1',
 'LADULT1',
 'COLGSEX',
 'NUMADULT',
 'LANDSEX',
 'NUMMEN',
 'NUMWOMEN',
 'RESPSLCT',
 'SAFETIME',
 'CTELNUM1',
 'CELLFON5',
 'CADULT1',
 'CELLSEX',
 'PVTRESD3',
 'CCLGHOUS',
 'CSTATE1',
 'LANDLINE',
 'HHADULT',
 'SEXVAR',
 'GENHLTH',
 'PHYSHLTH',
 'MENTHLTH',
 'POORHLTH',
 'PRIMINSR',
 'PERSDOC3',
 'MEDCOST1',
 'CHECKUP1',
 'EXERANY2',
 'BPHIGH6',
 'BPMEDS',
 'CHOLCHK3',
 'TOLDHI3',
 'CHOLMED3',
 'CVDINFR4',
 'CVDCRHD4',
 'CVDSTRK3',
 'ASTHMA3',
 'ASTHNOW',
 'CHCSCNCR',
 'CHCOCNCR',
 'CHCCOPD3',
 'ADDEPEV3',
 'CHCKDNY2',
 'DIABETE4',
 'DIABAGE3',
 'HAVARTH5',
 'ARTHEXER',
 'ARTHEDU',
 'LMTJOIN3',
 'ARTHDIS2',
 'JOINPAI2',
 'MARITAL',
 'EDUCA',
 'RENTHOM1',
 'NUMHHOL3',
 'NUMPHON3',
 'CPDEMO1B',
 'VETERAN3',
 'EMPLOY1',
 'CHILDREN',
 'INCOME3',
 'PREGNANT',
 'WEIGHT2',
 'HEIGHT3',
 'DEAF',
 'BLIND',
 'DECIDE',
 'DIFFWALK',
 'DIFF

In [14]:
cols_needed = [
    '_STATE',      # state code
    '_LLCPWT',     # survey weight
    '_BMI5',       # BMI
    '_AGEG5YR',    # age group
    '_SEX',        # sex
    '_EDUCAG',     # education group
    '_INCOMG1',    # income group
    '_RACE',       # race
]

brfss_small = brfss[cols_needed]

brfss_small.head()

,_STATE,_LLCPWT,_BMI5,_AGEG5YR,_SEX,_EDUCAG,_INCOMG1,_RACE
0,1.0,744.745531,1454.0,11.0,2.0,2.0,3.0,1.0
1,1.0,299.137394,NaN,10.0,2.0,4.0,9.0,2.0
2,1.0,587.862986,2829.0,11.0,2.0,2.0,2.0,2.0
3,1.0,1099.621573,3347.0,9.0,2.0,2.0,5.0,1.0
4,1.0,1711.825866,2873.0,12.0,1.0,1.0,2.0,7.0


In [15]:
brfss_small['BMI'] = brfss_small['_BMI5'] / 100

brfss_small[['BMI']].describe()

/tmp/ipykernel_702/3312212881.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brfss_small['BMI'] = brfss_small['_BMI5'] / 100


,BMI
count,391841.000000
mean,28.552265
std,6.551950
min,12.000000
25%,24.140000
50%,27.440000
75%,31.740000
max,99.330000


In [16]:
brfss_small['obese'] = (brfss_small['BMI'] >= 30).astype(int)

brfss_small['obese'].value_counts()

/tmp/ipykernel_702/2040490564.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brfss_small['obese'] = (brfss_small['BMI'] >= 30).astype(int)


,count
obese,
0,307388
1,131305


In [17]:
brfss_small = brfss_small[(brfss_small['_BMI5'] < 9000) & (brfss_small['_BMI5'] > 1200)]

brfss_small.shape

(391813, 10)

In [18]:
brfss_clean = brfss_small[
    (brfss_small['_AGEG5YR'] < 14) &
    (brfss_small['_SEX'] < 3) &
    (brfss_small['_EDUCAG'] < 5) &
    (brfss_small['_INCOMG1'] < 6)
]

brfss_clean.shape

(239678, 10)

In [19]:
import numpy as np

weighted_obesity_rate = np.average(
    brfss_clean['obese'],
    weights=brfss_clean['_LLCPWT']
)

weighted_obesity_rate

np.float64(0.3609551956750922)

In this step, the raw 2021 BRFSS microdata (438,693 respondents and 303 variables) was loaded and a subset of variables relevant to obesity modeling was extracted, including BMI, age group, sex, race, education, income, and survey weights. BMI values were converted from the BRFSS encoded format and an obesity indicator variable was created using the standard definition (BMI ≥ 30). Invalid BMI records and observations with missing or refused demographic responses were removed, resulting in a cleaned dataset of 239,678 respondents. A weighted national obesity prevalence was then computed using the BRFSS survey weights, yielding an estimate of approximately 36.1%, which aligns with CDC-reported national obesity rates. This validation confirms that the BRFSS dataset was correctly processed and provides a reliable foundation for subsequent modeling and integration with county-level ACS demographic data.